In [3]:
import os

# def load_json
def load_json(file_path):
    """
    Load a JSON file and return its content as a Python object.

    :param file_path: Path to the JSON file.
    :return: Python object representing the JSON data.
    """
    import json

    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

def all_filepaths_in_dir(root, endswith=None):
    file_paths = []
    for subdir, dirs, files in os.walk(root):
        for file in files:
            if endswith is None or file.endswith(endswith):
                file_paths.append(os.path.join(subdir, file))
    return file_paths

deepswe_with_path = "results/swe-atlas-qa/evolve-v2_chunk-swe-atlas-qa-0628-024710"
deepswe_without_path = "results/swe-atlas-qa/swe_atlas_without-0623-004601"

In [4]:
price = {}
price["uncached_input_token"] = 1 / 1_000_000
price["output_token"] = 2 / 1_000_000
price["cached_token"] = 0.02 / 1_000_000

def cal_api_cost(data):
    total_inp_tokens = data["final_metrics"]["total_prompt_tokens"]
    total_cache_tokens = data["final_metrics"]["total_cached_tokens"]
    total_out_tokens = data["final_metrics"]["total_completion_tokens"]

    total_uncached_input_tokens = total_inp_tokens - total_cache_tokens
    total_cost = (total_uncached_input_tokens * price["uncached_input_token"] +
                  total_cache_tokens * price["cached_token"] +
                  total_out_tokens * price["output_token"])
    return total_cost

cost_with = {}
for fn in all_filepaths_in_dir(deepswe_with_path, endswith="trajectory.json"):
    file_name = os.path.basename(fn)
    if file_name != "trajectory.json":
        continue
    # 从fn中提取case id
    case_id = os.path.basename(os.path.dirname(os.path.dirname(fn))).split('__')[0]
    
    data = load_json(fn)
    cost = cal_api_cost(data)
    cost_with[case_id] = cost

cost_without = {}
for fn in all_filepaths_in_dir(deepswe_without_path, endswith="trajectory.json"):
    file_name = os.path.basename(fn)
    if file_name != "trajectory.json":
        continue
    # 从fn中提取case id
    case_id = os.path.basename(os.path.dirname(os.path.dirname(fn))).split('__')[0]

    data = load_json(fn)
    cost = cal_api_cost(data)
    cost_without[case_id] = cost

print(cost_with.keys())
print(cost_without.keys())

joint_case_id = []
for id in cost_with:
    if id in cost_without:
        joint_case_id.append(id)

total_cost_with = sum([cost_with[id] for id in joint_case_id])
total_cost_without = sum([cost_without[id] for id in joint_case_id])

avg_cost_with = total_cost_with / len(joint_case_id)
avg_cost_without = total_cost_without / len(joint_case_id)

print(f"The average cost for with tools: {avg_cost_with}")
print(f"The average cost for without tools: {avg_cost_without}")
print(f"The reduction is: {(avg_cost_without-avg_cost_with)/avg_cost_without * 100} %")
print(f"The joint count is: {len(joint_case_id)}")

dict_keys(['task-6905333b74f22949d97baa15', 'task-6905333b74f22949d97baa28', 'task-6905333b74f22949d97ba9f2', 'task-6905333b74f22949d97ba9b8', 'task-6905333b74f22949d97baa1f', 'task-6905333b74f22949d97ba9d1', 'task-6905333b74f22949d97ba9b3', 'task-6905333b74f22949d97ba9e1', 'task-6905333b74f22949d97baa21', 'task-6905333b74f22949d97ba9d7', 'task-6905333b74f22949d97ba9e8', 'task-6905333b74f22949d97baa07', 'task-6905333b74f22949d97ba9c3', 'task-6905333b74f22949d97ba9f7', 'task-6905333b74f22949d97ba9c6', 'task-6905333b74f22949d97ba9cd', 'task-6905333b74f22949d97baa01', 'task-6905333b74f22949d97ba9d0', 'task-6905333b74f22949d97ba9ca', 'task-6905333b74f22949d97ba9f4', 'task-6905333b74f22949d97ba9bb', 'task-6905333b74f22949d97ba9cf', 'task-6905333b74f22949d97baa23', 'task-6905333b74f22949d97ba9ac', 'task-6905333b74f22949d97ba9dd', 'task-6905333b74f22949d97baa05', 'task-6905333b74f22949d97ba9e7', 'task-6905333b74f22949d97ba9af', 'task-6905333b74f22949d97ba9ad', 'task-6905333b74f22949d97baa22',

In [5]:
from src.tools.llm import LLM

In [3]:
# unset proxy
import os
os.environ['http_proxy'] = ''
os.environ['https_proxy'] = ''

llm = LLM('/home/fanmeihao/projects/CostReduce/_config/gpt5.5.yaml')
llm.query(system_prompt='answer the question', history='' , user_prompt='hello, how are you today?')

NotFoundError: 404 page not found